ReAct Agent using tools web search and calculator 

In [1]:
from langchain_anthropic import ChatAnthropic
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate
import math
from langchain_ollama import ChatOllama


# ===== LLM =====
llm = ChatOllama(
    model="llama3.2",
    temperature=0.7
)

# ===== TOOL 1: Web Search =====
search = DuckDuckGoSearchRun()
search.name = "web_search"
search.description = "Search the web for current information about any topic. Use this when you need facts, news, or information you don't already know."

# ===== TOOL 2: Calculator =====
@tool
def calculator(expression: str) -> str:
    """Use this to evaluate mathematical expressions.
    Input should be a valid Python math expression as a string.
    For example: '2 + 2', 'sqrt(16)', '15 * 8 / 3'
    """
    result = eval(expression, {"__builtins__": {}},
                     {"sqrt": math.sqrt, "pi": math.pi,
                      "sin": math.sin, "cos": math.cos,
                      "log": math.log, "abs": abs})
    return str(result)

# ===== TOOLS LIST =====
tools = [search, calculator]

# REACT Prompt Template
react_prompt = PromptTemplate.from_template("""Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}""")

# ===== AGENT =====
agent = create_react_agent(llm, tools, react_prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=5
)

print("Agent ready.")

Agent ready.


In [2]:
queries = [
    "What is the current population of Japan and what is that number divided by 1000?",
    "Search for the speed of light in metres per second and then calculate how far light travels in 10 seconds.",
    "What is the GDP of India in USD and what is the square root of that number in billions?"
]

for i, query in enumerate(queries):
    print(f"\n{'='*60}")
    print(f"QUERY {i+1}: {query}")
    print('='*60)
    result = agent_executor.invoke({"input": query})
    print(f"\nFINAL ANSWER: {result['output']}")


QUERY 1: What is the current population of Japan and what is that number divided by 1000?


> Entering new AgentExecutor chain...
Thought: To find the current population of Japan, I should search for recent data on Japan's population.
Action: web_search
Action Input: "current population of Japan"2 days ago ... With a population of almost 123 million as of 2026, it is the world's 11th most populous country. Tokyo is the country's capital and largest city. Japan. 日本. 16 Jun 2026 ... As of 2026, Japan has a total population of 122,427,731, ranking as the 12th most populous nation in the world. Japan has a net population change of ... 6 Apr 2026 ... What is the total population of Japan? ... In 2023, the estimated population of Japan was 124.4 million. In the chart, you can see how the population numbers have ... 7 Aug 2025 ... Japan's population is aging, shrinking and becoming slightly more diverse. The latest population figures show a total population falling by more than ... 15 Dec 20